# F2: Tokenization

### Overview
This notebook transforms the paragraph level F1 corpus into the two foundational tables for all downstream analysis: **TOKEN** (every word in every sentence in every paragraph) and **VOCAB** (the dictionary of distinct word-forms).

F2 extends the OHCO hierarchy two levels deeper, from paragraphs down to sentences, and from sentences own to individual tokens. Every word in Swift's writings has a unique address and a part-of-speech tag. 

### Pipeline
A single `tokenize()` function takes the DOC table and applies a two-stage split: NLTK's `sent_tokenize` breaks each paragraph into sentences, then `nltk.pos_tag` paired with `WhitespaceTokenizer` breaks each sentence into POS tagged tokens. Whitespace tokenization was chosen to keep multi-character units intact as single tokens, and punctuation that adheres to words is normalized away in the next step.

After TOKEN is built, a `term_str` column is added by lowercasing each `token_str` and stripping non-word characters via regex. This produces the normalized form that VOCAB uses aas its key. VOCAB itself is built from `TOKEN.term_str.value_counts()`, sorted alphabetically, and indexed by `term_id`. A binary `num` flag marks terms that begin with digits.

### Output
- **TOKEN** (`data/F2/TOKEN.csv`) one row per token, indexed by full OHCO, with columns `token_str`, `pos` and `term_str`.
- **VOCAB** (`data/F2/VOCAB.csv`) one row per distinct word-form, indexed by `term_id`, with columns `term_str`, `n` (frequency), and `num`

## Setup

In [69]:
import pandas as pd
import numpy as np
import re
import os
import nltk

F1_path = 'data/F1'
F2_path = 'data/F2'
os.makedirs(F2_path, exist_ok = True)

In [70]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [71]:
OHCO = ['work_id', 'part_num','chap_num','para_num','sent_num','token_num']

## Load F1 Data

In [72]:
LIB = pd.read_csv(f"{F1_path}/LIB.csv", index_col = OHCO[0])
DOC = pd.read_csv(f"{F1_path}/DOC.csv", index_col = OHCO[0:4])

print(f"LIB: {len(LIB)} works")
print(f"DOC: {len(DOC)} paragraphs")

DOC.head()

LIB: 49 works
DOC: 3579 paragraphs


para_str
work_id         part_num chap_num para_num                                                   
modest_proposal 1        1        1         It is a melancholy object to those, who walk t...
                                  2         I think it is agreed by all parties, that this...
                                  3         But my intention is very far from being confin...
                                  4         As to my own part, having turned my thoughts f...
                                  5         There is likewise another great advantage in m...

## Tokenize and Annotate

In [73]:
def tokenize(doc_df, OHCO=OHCO, remove_pos_tuple=False, ws=False):
    
    # Paragraphs to sentences
    df = doc_df.para_str.apply(lambda x: pd.Series(nltk.sent_tokenize(x))).stack().to_frame().rename(columns = {0:'sent_str'})

    # Sentences to tokens
    def word_tokenize(x):
        if ws:
            s = pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x)))
        else:
            s = pd.Series(nltk.pos_tag(nltk.word_tokenize(x)))
        return s
    
    df = df.sent_str.apply(word_tokenize).stack().to_frame().rename(columns = {0:'pos_tuple'})

    # grab info from tuple
    df['pos'] = df.pos_tuple.apply(lambda x: x[1])
    df['token_str'] = df.pos_tuple.apply(lambda x: x[0])
    if remove_pos_tuple:
        df = df.drop('pos_tuple', axis = 1)

    # add index
    df.index.names = OHCO

    return df

In [74]:
TOKEN = tokenize(DOC, ws = True)

TOKEN.head()

pos_tuple  \
work_id         part_num chap_num para_num sent_num token_num                     
modest_proposal 1        1        1        0        0                 (It, PRP)   
                                                    1                 (is, VBZ)   
                                                    2                   (a, DT)   
                                                    3          (melancholy, JJ)   
                                                    4              (object, NN)   

                                                               pos   token_str  
work_id         part_num chap_num para_num sent_num token_num                   
modest_proposal 1        1        1        0        0          PRP          It  
                                                    1          VBZ          is  
                                                    2           DT           a  
                                                    3           JJ  melancholy  
                                                    4           NN      object

## Build VOCAB

In [75]:
TOKEN['term_str'] = TOKEN['token_str'].str.lower().str.replace(r'[\W_]','',regex=True)
TOKEN.head()

pos_tuple  \
work_id         part_num chap_num para_num sent_num token_num                     
modest_proposal 1        1        1        0        0                 (It, PRP)   
                                                    1                 (is, VBZ)   
                                                    2                   (a, DT)   
                                                    3          (melancholy, JJ)   
                                                    4              (object, NN)   

                                                               pos  \
work_id         part_num chap_num para_num sent_num token_num        
modest_proposal 1        1        1        0        0          PRP   
                                                    1          VBZ   
                                                    2           DT   
                                                    3           JJ   
                                                    4           NN   

                                                                token_str  \
work_id         part_num chap_num para_num sent_num token_num               
modest_proposal 1        1        1        0        0                  It   
                                                    1                  is   
                                                    2                   a   
                                                    3          melancholy   
                                                    4              object   

                                                                 term_str  
work_id         part_num chap_num para_num sent_num token_num              
modest_proposal 1        1        1        0        0                  it  
                                                    1                  is  
                                                    2                   a  
                                                    3          melancholy  
                                                    4              object

In [76]:
VOCAB = (TOKEN.term_str.value_counts()
         .to_frame()
         .reset_index()
         .rename(columns = {'count':'n'})
         .sort_values('term_str')
         .reset_index(drop=True))
VOCAB.index.name = 'term_id'

In [77]:
VOCAB['num'] = VOCAB.term_str.str.match(r'\d+').astype('int')

VOCAB.sample(10)

,term_str,n,num
term_id,,,
10569,prettiest,1,0
5030,exposed,8,0
11813,run,60,0
3423,dargent,1,0
14929,war,59,0
12740,some,810,0
9472,opera,3,0
9386,ocean,5,0
8502,masquerading,1,0


In [78]:
TOKEN = TOKEN.drop('pos_tuple',axis=1)

TOKEN.to_csv(f"{F2_path}/TOKEN.csv")
VOCAB.to_csv(f"{F2_path}/VOCAB.csv")